[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/20_clustering/01_kmeans_from_scratch.ipynb)

# Bài 01 — K-Means from scratch

**🎯 Mục tiêu**
- Học không giám sát: tìm **nhóm** trong dữ liệu không có nhãn.
- Hàm mục tiêu của K-Means (inertia / WCSS) và thuật toán **Lloyd**: gán điểm ↔ cập nhật tâm — vì sao luôn hội tụ.
- Khởi tạo: random vs **k-means++**; cực tiểu địa phương và `n_init`.
- Ứng dụng: phân khúc khách hàng (nhớ scaling!), lượng tử hoá màu ảnh; Mini-Batch K-Means.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/20_clustering

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.datasets import load_sample_image, make_blobs
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

from ml_course.algorithms import KMeansScratch
from ml_course.datasets import make_customer_segments
from ml_course.visualization import PALETTE, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Bài toán

Cho $n$ điểm $\mathbf x_1, \dots, \mathbf x_n$, tìm $K$ tâm $\boldsymbol\mu_1, \dots, \boldsymbol\mu_K$ và phép gán $c_i \in \{1..K\}$ để tối thiểu **tổng bình phương khoảng cách trong cụm** (inertia, WCSS):

$$J = \sum_{i=1}^n \lVert\mathbf x_i - \boldsymbol\mu_{c_i}\rVert^2$$

Bài toán tối ưu toàn cục là NP-hard; **thuật toán Lloyd** tìm cực tiểu địa phương bằng cách luân phiên:
1. **Gán**: mỗi điểm về tâm gần nhất — $c_i = \arg\min_k \lVert\mathbf x_i - \boldsymbol\mu_k\rVert^2$ (tối ưu $J$ theo $c$ khi cố định $\mu$).
2. **Cập nhật**: mỗi tâm = trung bình các điểm của nó — $\boldsymbol\mu_k = \frac{1}{|C_k|}\sum_{i \in C_k}\mathbf x_i$ (tối ưu $J$ theo $\mu$ khi cố định $c$).

Mỗi bước **không làm tăng** $J$, và số cách gán là hữu hạn → luôn hội tụ (về cực tiểu địa phương).

In [ ]:
def kmeans(X, k, n_iter=100, seed=0, init="random"):
    r = np.random.default_rng(seed)
    if init == "random":
        centers = X[r.choice(len(X), k, replace=False)].copy()
    else:
        centers = kmeans_pp_init(X, k, r)
    history, inertias = [centers.copy()], []
    for _ in range(n_iter):
        d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(-1)       # (n, k) bình phương khoảng cách
        labels = d2.argmin(1)                                            # 1. gán
        inertias.append(d2[np.arange(len(X)), labels].sum())
        new = np.array([X[labels == j].mean(0) if np.any(labels == j) else centers[j] for j in range(k)])  # 2. cập nhật
        if np.allclose(new, centers):
            break
        centers = new
        history.append(centers.copy())
    return centers, labels, inertias, history


def kmeans_pp_init(X, k, r):
    """k-means++: tâm đầu ngẫu nhiên; tâm sau chọn với xác suất ∝ D(x)² — khoảng cách tới tâm gần nhất đã chọn."""
    centers = [X[r.integers(len(X))]]
    for _ in range(1, k):
        d2 = np.min(((X[:, None, :] - np.array(centers)[None]) ** 2).sum(-1), axis=1)
        centers.append(X[r.choice(len(X), p=d2 / d2.sum())])
    return np.array(centers)


X, y_true = make_blobs(n_samples=600, centers=4, cluster_std=0.9, random_state=3)
centers, labels, inertias, history = kmeans(X, 4, seed=5)
fig, axes = plt.subplots(1, 4, figsize=(22, 4.8))
for ax, step in zip(axes[:3], [0, 1, len(history) - 1]):
    c = history[step]
    lab = ((X[:, None, :] - c[None]) ** 2).sum(-1).argmin(1)
    ax.scatter(*X.T, c=[PALETTE[l] for l in lab], s=8, alpha=0.6)
    ax.scatter(*c.T, marker="X", s=250, color="k")
    ax.set_title(f"vòng {step}")
axes[3].plot(inertias, "o-")
axes[3].set_xlabel("vòng lặp")
axes[3].set_ylabel("inertia J")
axes[3].set_title("J giảm đơn điệu")
plt.show()
print(f"hội tụ sau {len(inertias)} vòng | inertia = {inertias[-1]:.1f}")

## 2. Khởi tạo và cực tiểu địa phương

Khởi tạo **random** đôi khi đặt hai tâm vào cùng một cụm thật → Lloyd mắc kẹt ở nghiệm tệ. **k-means++** chọn tâm trải đều (xác suất tỉ lệ $D(x)^2$) → nghiệm tốt hơn, và có bảo đảm $\mathbb E[J] \le O(\log K)\cdot J_{\text{opt}}$. Thêm vào đó, chạy `n_init` lần và giữ nghiệm có $J$ nhỏ nhất.

In [ ]:
X8, _ = make_blobs(n_samples=800, centers=8, cluster_std=0.6, random_state=11)
res = {"random": [], "k-means++": []}
for seed in range(50):
    for init in res:
        res[init].append(kmeans(X8, 8, seed=seed, init="random" if init == "random" else "pp")[2][-1])
best = min(min(v) for v in res.values())
for init, v in res.items():
    v = np.array(v)
    print(f"{init:10s}: inertia trung bình = {v.mean():8.1f} | tỉ lệ lần chạy đạt ≤ 1.01 × tốt nhất = {np.mean(v <= 1.01 * best):.0%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, init in zip(axes, ["random", "pp"]):
    seeds = np.argsort(res["random" if init == "random" else "k-means++"])[::-1]    # lần chạy TỆ NHẤT
    c, lab, J, _ = kmeans(X8, 8, seed=int(seeds[0]), init=init)
    ax.scatter(*X8.T, c=[PALETTE[l % len(PALETTE)] for l in lab], s=6, alpha=0.6)
    ax.scatter(*c.T, marker="X", s=200, color="k")
    ax.set_title(f"lần chạy tệ nhất với init = {init}: J = {J[-1]:.0f}")
plt.show()

## 3. Đối chiếu với scikit-learn

In [ ]:
sk = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)
ours_c, ours_l, ours_J, _ = kmeans(X, 4, seed=0, init="pp")
print(f"inertia: scratch = {ours_J[-1]:.2f} | sklearn = {sk.inertia_:.2f}")
ref = KMeansScratch(n_clusters=4, seed=0).fit(X)
print(f"KMeansScratch (src/ml_course): inertia = {ref.inertia_:.2f}")

## 4. Phân khúc khách hàng — kết quả phụ thuộc đơn vị đo

K-Means dùng khoảng cách Euclid → giống KNN (chương 08), feature có thang đo lớn áp đảo. Dữ liệu khách hàng gốc tình cờ có các feature cùng cỡ (thu nhập theo **nghìn** USD, điểm chi tiêu 1–100, tuổi). Chỉ cần đổi thu nhập sang **USD**:

In [ ]:
cust = make_customer_segments(n=1500, seed=2)
features = ["annual_income_musd", "spending_score", "age", "visits_per_month"]
X_k = cust[features].to_numpy()
X_usd = X_k.copy()
X_usd[:, 0] *= 1000                                                 # thu nhập: nghìn USD → USD
for name, Xc in [("thô (thu nhập: nghìn USD)", X_k), ("thô (thu nhập: USD)", X_usd), ("StandardScaler (bất kể đơn vị)", StandardScaler().fit_transform(X_usd))]:
    lab = KMeans(5, n_init=10, random_state=0).fit_predict(Xc)
    print(f"{name:32s} ARI so với phân khúc thật = {adjusted_rand_score(cust.true_segment, lab):.3f}")

Cùng một dữ liệu, chỉ đổi đơn vị, kết quả không scale sụp đổ (K-Means gần như chỉ chia theo thu nhập). Sau StandardScaler, kết quả **không phụ thuộc đơn vị**. (Ở đây dữ liệu thô với thu nhập theo nghìn USD tình cờ cho ARI cao hơn bản đã scale một chút — vì đơn vị đó vô tình gán trọng số phù hợp; đừng trông vào may mắn như vậy, và có thể chủ động gán trọng số feature theo hiểu biết miền.)

In [ ]:
km = KMeans(5, n_init=10, random_state=0).fit(StandardScaler().fit_transform(cust[features]))
cust["cluster"] = km.labels_
profile = cust.groupby("cluster")[features].mean().round(1)
profile["số khách"] = cust.cluster.value_counts().sort_index()
profile

**Diễn giải cụm** = nhìn trung bình (hoặc trung vị) từng feature trong cụm — "thu nhập cao, chi tiêu thấp, tuổi trung niên" → đặt tên phân khúc ("giàu thận trọng") → hành động marketing riêng. Đây là phần quan trọng nhất của clustering trong thực tế: cụm chỉ có giá trị khi **dùng được**.

## 5. Lượng tử hoá màu ảnh

Mỗi pixel là một điểm 3D (R, G, B). K-Means với $K$ = 16 → thay mỗi pixel bằng tâm cụm của nó → ảnh chỉ còn 16 màu (nén mạnh).

In [ ]:
img = load_sample_image("china.jpg") / 255.0
pixels = img.reshape(-1, 3)
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].imshow(img)
axes[0].set_title("gốc (~96 000 màu)")
for ax, k in zip(axes[1:], [2, 8, 32]):
    sample = pixels[rng.choice(len(pixels), 20000, replace=False)]           # fit trên mẫu con cho nhanh
    km_img = MiniBatchKMeans(k, random_state=0, n_init=3).fit(sample)
    ax.imshow(km_img.cluster_centers_[km_img.predict(pixels)].reshape(img.shape))
    ax.set_title(f"K = {k} màu")
for ax in axes:
    ax.axis("off")
plt.show()

## 6. Độ phức tạp và Mini-Batch K-Means

Mỗi vòng Lloyd: $O(n\,K\,d)$. Với hàng triệu điểm, **Mini-Batch K-Means** cập nhật tâm từ các lô nhỏ ngẫu nhiên — nhanh hơn nhiều, inertia chỉ hơi kém:

In [ ]:
Xbig, _ = make_blobs(n_samples=200_000, centers=10, n_features=10, random_state=0)
for name, model in [("KMeans", KMeans(10, n_init=3, random_state=0)), ("MiniBatchKMeans", MiniBatchKMeans(10, n_init=3, batch_size=2048, random_state=0))]:
    start = time.perf_counter()
    model.fit(Xbig)
    print(f"{name:16s} {time.perf_counter() - start:5.2f}s | inertia = {model.inertia_:,.0f}")

## 🧠 Kiểm tra nhanh

1. Vì sao Lloyd luôn hội tụ? Có bảo đảm hội tụ về nghiệm tối ưu toàn cục không?
2. k-means++ chọn tâm thứ hai thế nào?
3. Vì sao phải scale feature trước K-Means?
4. Độ phức tạp mỗi vòng lặp của K-Means?

## 📌 Tóm tắt

- K-Means tối thiểu inertia bằng Lloyd: gán ↔ cập nhật; hội tụ về cực tiểu địa phương.
- k-means++ + `n_init` giảm rủi ro nghiệm tệ.
- Scaling bắt buộc; diễn giải cụm bằng profile trung bình; Mini-Batch cho dữ liệu lớn.

## 📚 Tài liệu tham khảo

- Lloyd (1982) — *Least squares quantization in PCM*
- Arthur & Vassilvitskii (2007) — *k-means++: The Advantages of Careful Seeding*
- Vũ Hữu Tiệp — [K-means Clustering](https://machinelearningcoban.com/2017/01/01/kmeans/)